# Experiment 1: Llama vs Qwen at epoch 100

Plots closed-set top-1 watermark attribution accuracy against training-corpus size for the original Llama-on-Llama and Qwen-on-Qwen experiment-1 runs. The final corpus sizes differ by design: 63,800 for Llama and 50,000 for Qwen.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'results').is_dir():
    REPO_ROOT = Path.cwd().resolve().parents[1]
assert (REPO_ROOT / 'results').is_dir(), 'Run from the repository root or src/eval.'


In [ ]:
def load_top1(label, result_dir, corpus_sizes):
    rows = []
    for corpus_size in corpus_sizes:
        path = (REPO_ROOT / 'results' / result_dir / 'prefix_10' /
                str(corpus_size) / '32' / '100' / 'verification_closed.json')
        if not path.is_file():
            raise FileNotFoundError(path)
        result = json.loads(path.read_text())
        ranks = result['correct_ranks']
        rows.append({
            'model': label,
            'corpus_size': corpus_size,
            'top1_accuracy': sum(rank == 1 for rank in ranks) / len(ranks),
            'n_evaluated': len(ranks),
        })
    return rows

rows = load_top1('Llama 3.1 8B', 'experiment1', [100, 500, 1000, 5000, 10000, 63800])
rows += load_top1('Qwen 3.5 9B', 'experiment1-qwen', [100, 500, 1000, 5000, 10000, 50000])
df = pd.DataFrame(rows)
df.assign(top1_percent=100 * df.top1_accuracy)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for label, group in df.groupby('model', sort=False):
    ax.plot(group.corpus_size, 100 * group.top1_accuracy, marker='o', linewidth=2, label=label)

ax.set_xscale('log')
ax.set_xlabel('Training corpus size (log scale)')
ax.set_ylabel('Top-1 accuracy (%)')
ax.set_title('Experiment 1 at epoch 100')
ax.set_ylim(bottom=0)
ax.grid(True, which='both', alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()
